In [17]:
#!pip install -q ImageHash
#!pip install -q ftfy regex tqdm
#!pip install -q git+https://github.com/openai/CLIP.git

In [1]:
from pathlib import Path
from getpass import getpass
import shutil

import requests
from PIL import Image
import cv2
import imagehash

import torch
import clip
from sklearn.model_selection import train_test_split

In [10]:
# To keep the api key private, I used getpass to ask api key from you. 
Pexels_api_key = getpass("Enter Pexels api key: ")

In [11]:
# Here I am gonna define the api key and the url, then I download the pics 
pixels_urls = 'https://api.pexels.com/v1/search'
headers = {'Authorization':Pexels_api_key}
# Source: https://medium.com/%40minhlenguyen02/automating-image-search-and-download-with-the-pexels-api-module-7b50ad9e0ec1

In [ ]:
# This is the main dataset setting.
# If I wanna another ds later, this is the only part i need to edit. 
classes = {"bicycle":100, 'laptop':100, 'pizza':100}


# I save all downloaded images first.
# After the cleaning, I am gonna  use this folder to create the train and test split.
raw_ds = Path("Pexels-dataset/Raw-data")
raw_ds.mkdir(parents=True, exist_ok=True)

<center><font size="5">Load CLIP Model Once</font></center>


In [13]:
#  I got some pics that did not look like the class, so I use here CLIP to filter 
device = "cuda" if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load('ViT-B/32', device=device)
class_names = list(classes)
# Source: https://github.com/openai/CLIP

# I turn my class names into prompts for CLIP.
# Later, each downloaded image is compared with these prompt before I keep it. 
text = clip.tokenize([f"a photo of a {name}"for name in class_names]).to(device)

<center><font size="5">Creating  Dataset</font></center>

In [15]:
# In this cell, I preprocess the images

# if the cleaning becomes too strict or too weak, I can adjust them.
clip_confidence = 0.5
blur_min = 6
# I set a max page number 
max_pages = 25
# I create  function to compare the image with the class name. 
def comparing_img_VS_class(img, label):
    image= preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        scores = model(image, text)[0].softmax(dim=-1)[0]
# I keep the image only if the correct class win against the other prompt too.
    correct_img = class_names.index(label)
    return scores.argmax().item() == correct_img and scores[correct_img].item() >= clip_confidence

def not_blurry(path):
    img= cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.Laplacian(img, cv2.CV_64F).var()>=blur_min
# I used QWEN for debugging.
# Source: https://pyimagesearch.com/2015/09/07/blur-detection-with-opencv 

In [16]:
# In this cell, I download and clean the images
# Empty report, so I can print to see each class
report = []

for label, collecting_img in classes.items():
# one folder per class. 
    folder= raw_ds / label
    folder.mkdir(parents=True, exist_ok=True)

# I keep my clean ds from duplicates.
    accepted_img=[]
    too_small = miss_match = blur = accepted = duplicated = 0
    page = 1


    while len(accepted_img) < collecting_img and page <=max_pages:
# Because the Pexels allow max 80 images per page for having more the number of api call
        params = {'query':label, "per_page":80, 'page': page}
        response = requests.get(pixels_urls, headers=headers, params=params, timeout=20)
        response.raise_for_status() 
        photos = response.json().get("photos", [])

# I already kept enough clean images for this class
        for photo in photos:
            if len(accepted_img)>=collecting_img:
                break

  
# Pixels has different sizes of the same image, I choose the large.
# which is good balance, good enough quality, but not as heavy as original. 
            url = photo["src"]['large']
            path = folder/f"{photo['id']}.jpg"
           
# First I download the image and save it.
# After saving, I can open it with pillow and opencv 
            img_response = requests.get(url, timeout=20)
            img_response.raise_for_status()
            path.write_bytes(img_response.content)
            img = Image.open(path).convert('RGB')

# small image is not gonna be useful for training, so I remove them  now.
            if min(img.size)<200:
                  path.unlink(missing_ok=True)
                  too_small += 1
                  continue

# I calculated perceptual hash before accepting the image
            img_hash = imagehash.phash(img)
# Since my DS is not that big, I have to get very close images. 
            if any(img_hash-old_hash <=5 for old_hash in accepted_img):
                        path.unlink(missing_ok=True)
                        duplicated+= 1
                        continue
            


# After duplicate checking, I use clip to check the image meaning.
            if not comparing_img_VS_class(img, label):
              path.unlink(missing_ok=True)
              miss_match += 1
              continue

# I save the accepted image as jpeg
            if not not_blurry(path):
                path.unlink(missing_ok=True)
                blur += 1
                continue
            img.save(path, "JPEG", quality=95)
            accepted_img.append(img_hash) 
            accepted += 1
        page+= 1
    
# I save the result for this class so the next cell can print a clean summary.
    report.append([label, collecting_img, accepted, duplicated, miss_match, blur, too_small])
    print(f"{label}: Accepted={accepted}/{collecting_img}")
# Source: https://pypi.org/project/ImageHash
# For debugging, I used QWEN. 

bicycle: Accepted=100/100
laptop: Accepted=100/100
pizza: Accepted=100/100


In [17]:
for label, collecting_img, accepted, duplicated, miss_match, blur, too_small in report:
    print(f"{label}: accepted={accepted}, too small={too_small}, duplicated={duplicated}, miss match images={miss_match}, blur={blur}")

bicycle: accepted=100, too small=0, duplicated=2, miss match images=0, blur=0
laptop: accepted=100, too small=0, duplicated=2, miss match images=0, blur=0
pizza: accepted=100, too small=0, duplicated=2, miss match images=0, blur=0


In [20]:
# Save the class names for the annotating step.
class_names_path = raw_ds / "class_names.txt"
class_names_path.write_text("\n".join(class_names))

20